<a href="https://colab.research.google.com/github/joekelly211/masfi/blob/main/manuscript_revision_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports, directories and global functions

In [ ]:
# Define base directory
# Use '/content/drive/MyDrive/' for a personal drive
# Use '/gdrive/Shareddrives/' for a shared drive (must be created first)

# base_dir = "/gdrive/Shareddrives/masfi"
base_dir = '/content/drive/MyDrive/masfi'

# Mount Google Drive
from google.colab import drive
import os
import sys
if base_dir.startswith('/gdrive/Shareddrives/'):
  drive.mount('/gdrive', force_remount=True)
elif base_dir.startswith('/content/drive/MyDrive/'):
  drive.mount('/content/drive', force_remount=True)
  os.makedirs(base_dir, exist_ok=True)
else: print("Create a base_dir beginning with '/gdrive/Shareddrives/' or '/content/drive/MyDrive/'.")

_path_to_add = os.path.realpath(base_dir)
if _path_to_add not in sys.path:
    sys.path.append(_path_to_add)

In [ ]:
# Capture outputs
%%capture
# Installs and upgrades
!pip install astropy
!pip install geopandas
!pip install kaleido==0.2.1
!pip install tabulate
!pip install xgboost

In [ ]:
# Imports
from astropy.convolution import convolve, Gaussian2DKernel
import ast
from concurrent.futures import ThreadPoolExecutor
try: import cupy # Only works on GPU runtime
except: None
import gc
import geopandas as gpd
from google.colab import runtime
import ipywidgets as widgets
import json
import kaleido
import numpy as np
from os import makedirs
from os.path import basename, exists, join
from osgeo import gdal, ogr
gdal.UseExceptions()
import pandas as pd
import plotly.graph_objects as go
from scipy.ndimage import distance_transform_edt
from scipy.spatial import cKDTree
from scipy.stats import spearmanr
from shapely.geometry import box
import shutil
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from tabulate import tabulate
import warnings
import xgboost as xgb

In [ ]:
# Define directories
areas_dir = join(base_dir, "1_areas")
polygons_dir = join(areas_dir, "polygons")

features_dir = join(base_dir, "3_features")
feature_binary_dir = join(features_dir, "binary")
feature_edge_effects_dir = join(features_dir, "binary_edge_effects")

# 4_datasets directories
datasets_dir = join(base_dir, "4_datasets")
datasets_final_dir = join(datasets_dir, "final")

models_dir = join(base_dir, "5_models")
scenarios_dir = join(base_dir, "6_scenarios")

# Define and create statistics directories
statistics_dir = join(base_dir, "9_statistics")
sample_polygons_dir = join(statistics_dir, "sample_polygons")

# Tests directory
tests_dir = join(base_dir, "ms_revision_tests")

# Create directories
makedirs(tests_dir, exist_ok=True)

In [ ]:
# Global function: read raster as array
def read_raster_as_array(path):
    ds = gdal.Open(path)
    arr = ds.ReadAsArray()
    ds = None
    return arr

# Global function: export an array as a .tif
template_tif_path = join(areas_dir, "template.tif")
nodatavalue = -11111
compress = True
def export_array_as_tif(input_array, output_tif, template=template_tif_path, nodatavalue=nodatavalue, compress=compress, dtype=gdal.GDT_Float32):
    template_ds = gdal.Open(template)
    template_band = template_ds.GetRasterBand(1)
    template_dimensions, template_projection = template_ds.GetGeoTransform(), template_ds.GetProjection()
    if compress: options = ['COMPRESS=ZSTD', 'ZSTD_LEVEL=1'] # Good speed / size ratio
    else: options = []
    if input_array.dtype == 'int16': dtype = gdal.GDT_Int16
    driver = gdal.GetDriverByName("GTiff").Create(output_tif, template_band.XSize, template_band.YSize, 1, dtype, options=options)
    driver.GetRasterBand(1).WriteArray(input_array)
    driver.GetRasterBand(1).SetNoDataValue(nodatavalue)
    driver.SetGeoTransform(template_dimensions)
    driver.SetProjection(template_projection)
    template_ds = driver = None

# Global function: burn a polygon to raster (from 3a_features_scenario.ipynb)
def burn_polygon_to_raster(raster_path, polygon_path, fixed=True, fixed_value=1, column_name=None, all_touched=True):
    raster = vector = None
    try:
        raster = gdal.Open(raster_path, gdal.GA_Update)
        vector = ogr.Open(polygon_path)
        if not raster or not vector:
            raise ValueError("Cannot open input files")
        layer = vector.GetLayer()
        options = ["ALL_TOUCHED=TRUE"] if all_touched else []
        if fixed:
            gdal.RasterizeLayer(raster, [1], layer, burn_values=[fixed_value], options=options)
        else:
            attr_name = column_name or layer.GetLayerDefn().GetFieldDefn(0).GetName()
            options.append(f"ATTRIBUTE={attr_name}")
            gdal.RasterizeLayer(raster, [1], layer, options=options)
    finally:
        if raster: raster.FlushCache()
        raster = vector = None

# Global function: edge effects (from 3a_features_scenario.ipynb)
# Provides spatial awareness analogous to CNN receptive fields for tabular models
# Data_type: 'binary' or 'continuous'.
cell_size_y_path = join(areas_dir, 'cell_size_y.tif')
cell_size_x_path = join(areas_dir, 'cell_size_x.tif')
def edge_effects(array, data_type, cell_size_x_path, cell_size_y_path, threshold_metres):
    # Determine pixel size from cell size rasters.
    cell_size_x = gdal.Open(cell_size_x_path)
    cell_size_x_array = cell_size_x.ReadAsArray()
    cell_size_x = None
    cell_size_y = gdal.Open(cell_size_y_path)
    cell_size_y_array = cell_size_y.ReadAsArray()
    cell_size_y = None
    mean_cell_resolution = np.mean([np.mean(cell_size_x_array), np.mean(cell_size_y_array)])
    # Maximum pixel distance for kernel extent.
    max_pixel_distance = threshold_metres / mean_cell_resolution
    # 2D Gaussian weight distribution follows chi-squared with df=2.
    # Cumulative probability in radius r: P = 1 - exp(-r² / 2σ²).
    # Solving for r at P=0.95: r = σ * sqrt(-2 * ln(0.05)) ≈ 2.45σ.
    # Setting r = max_pixel_distance ensures 95% of kernel weight falls within threshold.
    gaussian_stdev = max_pixel_distance / 2.45
    kernel_radius = int(np.ceil(max_pixel_distance))
    kernel_size = 2 * kernel_radius + 1
    # Gaussian kernel for spatial weighting.
    kernel = Gaussian2DKernel(x_stddev=gaussian_stdev, y_stddev=gaussian_stdev,
                              x_size=kernel_size, y_size=kernel_size)
    # Circular mask enforces ecological threshold as hard boundary.
    # Square kernels would include pixels beyond threshold at corners.
    y, x = np.ogrid[:kernel_size, :kernel_size]
    centre = kernel_radius
    distance_from_centre = np.sqrt((x - centre)**2 + (y - centre)**2)
    circular_mask = distance_from_centre <= max_pixel_distance
    # Apply mask and renormalise to sum to 1.
    # Renormalisation ensures consistent weighting after truncation.
    kernel_array = kernel.array.copy()
    kernel_array[~circular_mask] = 0
    kernel_array /= kernel_array.sum()
    # Gaussian smoothing captures local spatial context.
    # For binary: represents local class density in threshold.
    # For continuous: represents local weighted mean in threshold.
    # boundary='extend' extrapolates edge values beyond raster extent.
    smoothed = convolve(array.astype(float), kernel_array, boundary='extend')
    if data_type == 'continuous': return smoothed # Without rounding
    if data_type == 'binary': smoothed = np.round(smoothed, 2) # Round
    # Binary data: compute signed distance to class boundary.
    # Euclidean distance transform gives centre-to-centre pixel distance.
    dist_from_ones = distance_transform_edt(array == 0)
    dist_from_zeros = distance_transform_edt(array == 1)
    # Convert to distance from pixel centre to class boundary.
    # Class boundary lies between adjacent pixels of different classes.
    # Subtracting 0.5 pixels approximates centre-to-boundary distance.
    # Sign encodes class membership: positive = class 1, negative = class 0.
    # Magnitude encodes proximity to boundary (edge effects zone).
    signed_distance = np.where(
        array == 1,
        np.maximum(dist_from_zeros - 0.5, 0) * mean_cell_resolution,
        -np.maximum(dist_from_ones - 0.5, 0) * mean_cell_resolution
    )
    # Cap at threshold: pixels beyond are interior, not edge-influenced.
    # Round to integer metres for cleaner feature representation.
    signed_distance = np.round(np.clip(signed_distance, -threshold_metres, threshold_metres)).astype(np.int16)
    return signed_distance, smoothed

# Global function: Sample raster values (from 4_datasets.ipynb)
def sample_raster_values(pd_dataframe, raster_path, geom_x, geom_y, feature=False, n_threads=1):
    # Derive column name from filename
    raster_name = raster_path.split('/')[-1][:-4]
    if feature: raster_name = 'fea_' + raster_name
    # Load raster and extract metadata
    raster = gdal.Open(raster_path)
    band = raster.GetRasterBand(1)
    geotransform = raster.GetGeoTransform()
    raster_array = band.ReadAsArray()
    nodata = band.GetNoDataValue()
    rows, cols = raster_array.shape
    fill_value = nodata if nodata is not None else np.nan
    # Initialise output array with nodata
    sampled_values = np.full(len(geom_x), fill_value, dtype=raster_array.dtype)
    # Worker function for threaded sampling
    def sample_chunk(start, end):
        x_idx = ((geom_x[start:end] - geotransform[0]) / geotransform[1]).astype(int)
        y_idx = ((geom_y[start:end] - geotransform[3]) / geotransform[5]).astype(int)
        valid = (x_idx >= 0) & (x_idx < cols) & (y_idx >= 0) & (y_idx < rows)
        local_values = np.full(end - start, fill_value, dtype=raster_array.dtype)
        local_values[valid] = raster_array[y_idx[valid], x_idx[valid]]
        sampled_values[start:end] = local_values
    # Split points into chunks and process in parallel
    n_points = len(geom_x)
    chunk_size = (n_points + n_threads - 1) // n_threads
    chunk_ranges = [(i, min(i + chunk_size, n_points)) for i in range(0, n_points, chunk_size)]
    with ThreadPoolExecutor(max_workers=n_threads) as executor:
        executor.map(lambda r: sample_chunk(*r), chunk_ranges)
    # Assign to dataframe and release resources
    pd_dataframe[raster_name] = sampled_values
    raster = band = None

# Global function: rasterise polygon to row / column indices and cell areas in hectares
# (from 9_statistics.ipynb 'Select sample polygons')
# Pixel centre intersection (all_touched=False) avoids double-counting.
cell_area_path = join(areas_dir, "cell_area.tif")
def polygon_pixel_indices(polygon_path):
    # Load cell area array
    cell_area_ds = gdal.Open(cell_area_path)
    cell_area_array = cell_area_ds.ReadAsArray().astype('float64') / 10000
    zeros_array = np.zeros(cell_area_array.shape, dtype='int16')
    cell_area_ds = None
    # Temporary raster in tests directory, burned with single polygon
    temp_raster_path = join(tests_dir, 'temp_burn.tif')
    export_array_as_tif(zeros_array, temp_raster_path)
    temp_ds = gdal.Open(temp_raster_path, gdal.GA_Update)
    band = temp_ds.GetRasterBand(1)
    vector = ogr.Open(polygon_path)
    layer = vector.GetLayer()
    gdal.RasterizeLayer(temp_ds, [1], layer, burn_values=[1], options=[])
    temp_ds.FlushCache()
    vector = None
    # Extract indices and cell areas
    rows, cols = np.where(band.ReadAsArray() == 1)
    cell_area_ha = cell_area_array[rows, cols]
    temp_ds = None
    os.remove(temp_raster_path)
    return rows, cols, cell_area_ha

# Select model

In [ ]:
# Select a model
model_exists = False
for subdir, dirs, files in os.walk(models_dir):
  for file in files:
    if file == 'model.json':
      print(f'selected_model = "{subdir.split(f"{models_dir}/",1)[1]}"')
      model_exists = True
if not model_exists: print("No model exists.")

In [ ]:
selected_model = "agbd_v2_1_260206_133525"

# Define model directories (from 6_scenarios.ipynb 'Select model')
selected_model_dir = join(models_dir,selected_model)
selected_model_shap_dir = join(selected_model_dir, "shap")
selected_model_json = join(selected_model_dir, "model.json")
selected_model_descr_dir = join(selected_model_dir, "model_description.json")
selected_model_dataset_path = join(selected_model_dir, f"{selected_model}.pkl")

# Read description for model dataset attributes
with open(join(selected_model_dir,"model_dataset_description.json")) as model_dataset_description_json:
  model_dataset_description = json.load(model_dataset_description_json)
selected_target = model_dataset_description["selected_target"]
covariates_renamed = model_dataset_description["covariates_renamed"]
selected_features = model_dataset_description["selected_features"] + model_dataset_description["covariates_renamed"]
categorical_features_mappings = model_dataset_description["categorical_features_mappings"]
categorical_columns = model_dataset_description["categorical_columns"]

covariates = [covariate[4:] for covariate in covariates_renamed]

# Create scenarios model directory
scenarios_model_dir = join(scenarios_dir, selected_model)

# Create subdirectories
scenario_masks_dir = join(scenarios_model_dir, "scenario_masks")
tile_templates_dir = join(scenarios_model_dir, 'tile_templates')
feature_stacks_dir = join(scenarios_model_dir, "feature_stacks")

# Load model dataset (from 5_models.ipynb 'Model parameters')
# Read from selected model directory
model_dataset = pd.read_pickle(selected_model_dataset_path)

# WIP categorise_target
categorise_target = False

XGBPredictor = xgb.XGBRegressor
eval_metric = 'rmse'
metric = 'rmse'

# Define optimal values dictionary
optimal_values = {
    "r2": "max", "me": "min", "rmse": "min", "rrmse": "min",
    "accuracy": "max", "precision": "max", "recall": "max", "f1": "max",
    "roc_auc": "max", "roc_auc_ovr": "max", "tss": "max", "mcc": "max",
    "balanced_accuracy": "max", "average_precision": "max", "cohen_kappa": "max",
    "specificity": "max", "logloss": "min", "mlogloss": "min",
    "brier_score": "min"
}

# Define x and y axes for training
model_dataset_x = model_dataset[selected_features].copy()
model_dataset_y = model_dataset[selected_target].copy()

# Create dictionary for optimal value and define the weighted score
def def_score_weighted(mean_test, mean_training, metric_name=None):
    if metric_name is None:
        metric_name = metric
    opt = optimal_values[metric_name]
    if opt == "max":
        return mean_test - 0.5*max(0, mean_training - mean_test)
    return mean_test + 0.5*max(0, mean_test - mean_training)

# Avoids issues using dataframe from CPU
xgb.set_config(verbosity=0, use_rmm=True)

# Reload hyperparameters (from 5_models.ipynb 'Final model', 'Train and export with full dataset')
with open(selected_model_descr_dir) as model_description_json:
  model_description = json.load(model_description_json)
final_hyperparameters = ast.literal_eval(model_description["hyperparameters"])

# Hyperparameters of test models
test_hyperparameters = dict(final_hyperparameters)

# Remove early stopping and replace with mean n_estimators
if "early_stopping_rounds" in final_hyperparameters:
  final_hyperparameters = {k:v for k, v in final_hyperparameters.items() if k != "early_stopping_rounds"}
  final_hyperparameters["n_estimators"] = round(model_description["n_estimators mean"])

# New test directories for the analyses
tests_model_dir = join(tests_dir, selected_model)
spatial_dir = join(tests_model_dir, "spatial_independence")
neutral_dir = join(tests_model_dir, "neutral_covariates")
holdout_dir = join(tests_model_dir, "old_growth_holdout")
makedirs(tests_model_dir, exist_ok=True)
makedirs(spatial_dir, exist_ok=True)
makedirs(neutral_dir, exist_ok=True)
makedirs(holdout_dir, exist_ok=True)

print(f"{selected_model} loaded with {len(model_dataset):,} footprints and {len(selected_features)} features.")
print(f"Final model n_estimators: {final_hyperparameters['n_estimators']}")

In [ ]:
# Exact footprint coordinates. Latitude and longitude features rounded to 3 decimals
# (~110 m) in 4_datasets.ipynb 'Finalise dataset', so footprint geometry is loaded from
# final dataset, which 5_models.ipynb normally drops on import
for file in os.listdir(datasets_final_dir):
  if file.endswith(".pkl"): print(f'source_dataset = "{file}"')

In [ ]:
source_dataset = "agbd_v2_1.pkl"

# Match footprints to model dataset footprints by shot number
source_dataset_read = pd.read_pickle(join(datasets_final_dir, source_dataset))[['tar_shot_number', 'tar_geometry']]
footprint_geometry = source_dataset_read.set_index('tar_shot_number')['tar_geometry'].reindex(model_dataset['shot_number'])
assert footprint_geometry.notna().all(), "Not all model dataset footprints found in the source dataset."
footprint_x = np.array([g.x for g in footprint_geometry])
footprint_y = np.array([g.y for g in footprint_geometry])
source_dataset_read = footprint_geometry = None
print(f"Coordinates loaded for {len(footprint_x):,} footprints.")

# Sample polygons

In [ ]:
# Select sample area polygons. This should be a single .gpkg with the field 'name' differentiating polygons.
sample_polygons = []
for geopackage in os.listdir(sample_polygons_dir):
  sample_polygons.append(geopackage)
if len(sample_polygons) == 0:
  print(f"No sample areas found. Upload .gpkg polygons to {sample_polygons_dir}")
else:
  for sample_polygon in sample_polygons: print(f"selected_sample_polygons = '{sample_polygon}'")

In [ ]:
selected_sample_polygons = 'final_sample_polygons.gpkg'

# Define and create sample polygons directory (from 9_statistics.ipynb 'Select sample polygons')
selected_sample_polygons_dir = join(sample_polygons_dir, selected_sample_polygons)
selected_sample_polygons_gpkg = gpd.read_file(selected_sample_polygons_dir)
# Add sample polygon statistics to the tests directory
sample_polygons_statistics_dir = join(tests_model_dir, selected_sample_polygons[:-5])
makedirs(sample_polygons_statistics_dir, exist_ok=True)

# Rasterise each sample polygon, save row/col indices and cell areas.
# Pixel centre intersection (all_touched=False) avoids double-counting.

# Structure of .npz files:
#   rows, cols: 1D int arrays of pixel indices in polygon
#   cell_area_ha: 1D float array of pixel areas in hectares
# Later statistics sections load these arrays, extract raster values at [rows, cols],
# filter nodata, then compute area-weighted statistics using cell_area_ha.

# Sort polygons alphabetically instead of geopackage order
sort_polygons_alphabetically = False

# Reprocess polygons that already have an .npz
overwrite_existing = False

polygon_indices_dir = join(sample_polygons_statistics_dir, 'polygon_indices')
makedirs(polygon_indices_dir, exist_ok=True)

polygon_names = [row['name'] for _, row in selected_sample_polygons_gpkg.iterrows()]
if sort_polygons_alphabetically:
    polygon_names = sorted(polygon_names)

# Zero-padded prefix width based on polygon count
n_polygons = len(polygon_names)
prefix_width = len(str(n_polygons))

# Check existing outputs before any raster work
polygon_outputs = [(i, name, join(polygon_indices_dir, f'{str(i).zfill(prefix_width)}_{name}.npz'))
                   for i, name in enumerate(polygon_names)]
if overwrite_existing:
    polygons_to_process = polygon_outputs
else:
    polygons_to_process = [entry for entry in polygon_outputs if not exists(entry[2])]
n_existing = sum(1 for entry in polygon_outputs if exists(entry[2]))

if not polygons_to_process:
    print(f"All {n_polygons} polygon index arrays already exist in {polygon_indices_dir}")
else:
    if overwrite_existing and n_existing:
        print(f"Overwriting {n_existing}/{n_polygons} existing polygon index arrays")
    elif n_existing:
        print(f"{n_existing}/{n_polygons} polygon index arrays already exist, processing {len(polygons_to_process)}")

    # Load cell area array
    cell_area_ds = gdal.Open(cell_area_path)
    cell_area_array = cell_area_ds.ReadAsArray().astype('float64') / 10000
    zeros_array = np.zeros(cell_area_array.shape, dtype='int16')
    cell_area_ds = None

    temp_raster_path = join(polygon_indices_dir, 'temp_burn.tif')
    export_array_as_tif(zeros_array, temp_raster_path)

    temp_ds = gdal.Open(temp_raster_path, gdal.GA_Update)
    band = temp_ds.GetRasterBand(1)

    progress_max = len(polygons_to_process)
    progress_index = 0
    progress_label = widgets.Label(f"Polygon progress: {progress_index}/{progress_max}")
    display(progress_label)

    for i, polygon_name, npz_path in polygons_to_process:
        # Burn polygon with unique value
        vector = ogr.Open(selected_sample_polygons_dir)
        layer = vector.GetLayer()
        layer.SetAttributeFilter(f"name = '{polygon_name}'")
        gdal.RasterizeLayer(temp_ds, [1], layer, burn_values=[i + 1], options=[])
        temp_ds.FlushCache()
        vector = None

        # Extract indices and cell areas
        rows, cols = np.where(band.ReadAsArray() == i + 1)
        cell_area_ha = cell_area_array[rows, cols]
        np.savez(npz_path, rows=rows, cols=cols, cell_area_ha=cell_area_ha)

        progress_index += 1
        progress_label.value = f"Polygon progress: {progress_index}/{progress_max}"

    temp_ds = None
    os.remove(temp_raster_path)

    print(f"Polygon index arrays saved to {polygon_indices_dir}")

# Load polygon data from .npz files
polygon_data = {}
for npz_file in sorted(os.listdir(polygon_indices_dir)):
    if npz_file.endswith('.npz'):
        name = npz_file.split('_', 1)[1][:-4]
        data = np.load(join(polygon_indices_dir, npz_file))
        polygon_data[name] = {
            'rows': data['rows'],
            'cols': data['cols'],
            'cell_area_ha': data['cell_area_ha']
        }

# Sample polygon names, which are used to choose management units in later
print("# Sample polygon names")
for polygon_name in polygon_data: print(f'  "{polygon_name}",')

# Reserve GEDI footprints

In [ ]:
# Training footprints within each sample polygon - their count, density and AGBD distribution
# This describes the training data, not polygon AGBD, which depends on footprint sampling and
# machine learning predictions for the entire area
footprints_dir = join(tests_model_dir, "reserve_footprints")
makedirs(footprints_dir, exist_ok=True)

# Training footprints with target AGBD as points at exact coordinates
footprints_gdf = gpd.GeoDataFrame({'agbd': model_dataset_y.to_numpy(dtype=np.float64)},
                                  geometry=gpd.points_from_xy(footprint_x, footprint_y), crs="EPSG:4326")

# Dissolve sample polygons by name, multi-parts are summarised together
units_gdf = selected_sample_polygons_gpkg[['name', 'geometry']].to_crs("EPSG:4326").dissolve(by='name').reset_index()

# Number of footprints in each polygon
footprints_joined = gpd.sjoin(footprints_gdf, units_gdf, predicate='within')

# Summarise in sample polygon order
unit_order = list(dict.fromkeys(selected_sample_polygons_gpkg['name']))

# 2024 forest area, as footprints were filtered to forest
# (from 9_statistics.ipynb 'Forest cover')
forest_array = read_raster_as_array(join(areas_dir, "masks", "mask_forest_2024.tif"))
forest_area_km2 = [polygon_data[name]['cell_area_ha'][forest_array[polygon_data[name]['rows'], polygon_data[name]['cols']] == 1].sum(dtype='float64') / 100
                   for name in unit_order]
forest_array = None

# 5th and 95th percentiles of AGBD
def percentile_5(agbd):
    return agbd.quantile(0.05)
def percentile_95(agbd):
    return agbd.quantile(0.95)

footprint_summary = footprints_joined.groupby('name')['agbd'].agg(
    ['count', 'median', percentile_5, percentile_95]).reindex(unit_order)
footprint_summary.columns = ['Footprints', 'Median AGBD', '5th percentile AGBD', '95th percentile AGBD']
footprint_summary.insert(1, 'Forest area (km2)', forest_area_km2)
footprint_summary.insert(2, 'Footprints per km2 forest', footprint_summary['Footprints'] / footprint_summary['Forest area (km2)'])
footprint_summary.to_csv(join(footprints_dir, "reserve_footprint_summary.csv"))

print(tabulate(footprint_summary.round(1), headers='keys', tablefmt='simple'))
print(f"\nReserve footprint summary went to {footprints_dir}")

# KFold cross-validation

In [ ]:
# The number of k-folds affects the proportion of data used for training,
# with the complementary proportion used for validation / testing.
# 10 splits = 90 % training, 5 = 80 %, 4 = 75 %, 3 = 67 %, 2 = 50 %.

# Input the number of k-folds to generate train/valid splits.
n_splits = 10

# Number of folds to be used for HPO validation.
# Remaining folds will be used for final testing.
n_hpo_splits = 2

assert n_hpo_splits < n_splits, "n_hpo_splits must be less than n_splits"

# Stratified k-fold ensures each fold has representative target distribution
# For regression, target is binned into quantiles using Sturges' rule
n_bins = max(5, int(np.ceil(1 + np.log2(len(model_dataset_y)))))
y_binned = pd.qcut(model_dataset_y, q=n_bins, labels=False, duplicates='drop')
kf_split = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=1).split(model_dataset_x, y_binned))

train_pct = (n_splits - 1) / n_splits * 100
print(f"Created {n_splits} stratified folds ({train_pct:.0f}% train, {100 - train_pct:.0f}% validation per fold)")
print(f"HPO uses {n_hpo_splits} folds, {n_splits - n_hpo_splits} reserved for final testing")
print(f"Regression target binned into {len(np.unique(y_binned))} quantiles for stratification")

In [ ]:
# Return a dictionary full of metrics for calculating accuracy
def calculate_metrics(validation, prediction, prediction_proba=None):
    # Regression metrics
    validation = validation.astype("float32")
    prediction = prediction.astype("float32")
    metrics = {
        "r2": r2_score(validation, prediction),
        "me": np.mean(np.array(validation).squeeze() - np.array(prediction).squeeze()),
        "rmse": root_mean_squared_error(validation, prediction),
        "rrmse": (root_mean_squared_error(validation, prediction) / np.mean(validation)) * 100 if np.mean(validation) != 0 else np.nan,
        "optimal_value": optimal_values
    }
    return metrics

def run_kfold(model_dataset, kf_split, predictor, verbose=False, export_test_model=False, test_model_dir=None):
    metrics = []

    # Junk data is used to determine column names by seeing what metrics get calculated
    junk_data = pd.Series([0, 1, 0, 1, 0, 1])
    junk_proba = np.array([0.5] * 6) if categorise_target else None
    junk_metrics = calculate_metrics(junk_data, junk_data, junk_proba if categorise_target else None)

    cols = []
    for m in [k for k in junk_metrics.keys() if k != 'optimal_value']:
        cols += [f"score_training ({m})", f"score_test ({m})", f"score_difference ({m})", f"score_weighted ({m})"]
    cols.append("n_estimators")

    fold_index = 1
    for train_index, valid_index in kf_split:
        x_train = model_dataset.loc[train_index][selected_features]
        x_valid = model_dataset.loc[valid_index][selected_features]
        y_train = model_dataset.loc[train_index][selected_target]
        y_valid = model_dataset.loc[valid_index][selected_target]

        dtrain = xgb.DMatrix(x_train, y_train, enable_categorical=True)
        dvalid = xgb.DMatrix(x_valid, y_valid, enable_categorical=True)

        params = predictor.get_params()
        params['eval_metric'] = eval_metric

        # Sklearn wrapper params handled separately: n_estimators via num_boost_round,
        # Early_stopping_rounds via callback, enable_categorical via DMatrix
        for key in ['n_estimators', 'enable_categorical', 'missing', 'early_stopping_rounds']:
            params.pop(key, None)

        early_stopping = xgb.callback.EarlyStopping(
            rounds=predictor.early_stopping_rounds,
            metric_name=eval_metric,
            data_name='validation_1'
        )

        evals_result = {}
        model = xgb.train(params,
                          dtrain,
                          num_boost_round=predictor.n_estimators,
                          evals=[(dtrain, 'training'), (dvalid, 'validation_1')],
                          callbacks=[early_stopping],
                          evals_result=evals_result,
                          verbose_eval=False)

        # Get predictions
        training_prediction_proba = model.predict(dtrain)
        validation_prediction_proba = model.predict(dvalid)

        training_metrics = calculate_metrics(y_train, training_prediction_proba)
        validation_metrics = calculate_metrics(y_valid, validation_prediction_proba)

        # Build metrics row
        metrics_row = []
        for m in [k for k in training_metrics.keys() if k != 'optimal_value']:
            score_training = training_metrics[m]
            score_test = validation_metrics[m]
            score_difference = score_training - score_test
            score_weighted = def_score_weighted(score_test, score_training, m)
            metrics_row += [score_training, score_test, score_difference, score_weighted]

        metrics_row.append(len(evals_result['validation_1'][eval_metric]))
        metrics.append(metrics_row)

        if export_test_model:
            makedirs(test_model_dir, exist_ok=True)
            model_name = f"model_test_fold_{fold_index}"
            metrics_dir = join(test_model_dir, f"{model_name}_metrics.csv")
            model.save_model(join(test_model_dir, f"{model_name}.json"))
            df_fold_metrics = pd.DataFrame(metrics, columns=cols)
            df_fold_metrics.to_csv(metrics_dir)
            print(f"Model export complete for test fold {fold_index}.")

        if verbose:
            score_training = training_metrics[metric]
            score_test = validation_metrics[metric]
            score_difference = score_training - score_test
            score_weighted = def_score_weighted(score_test, score_training)
            print(f"Validation {metric}: {score_test}")
            print(f"Training {metric}: {score_training}")
            print(f"{metric} difference: {score_difference}")
            print(f"Weighted {metric}: {score_weighted}")
            print(f"--------------------")

        fold_index += 1

    df_kfold_metrics = pd.DataFrame(metrics, columns=cols)
    return df_kfold_metrics


# Spatial independence

## Setup

In [ ]:
# Tests spatial independence of cross-validation folds. Random folds (which were AGBD-stratified)
# are compared with spatially blocked folds, using same hyperparameters and test folds.

# Test model (from 5_models.ipynb 'Final model','Define and test')
# For exported test models of the random folds
test_model_dir = join(selected_model_dir,"model_test")

# Random folds not set aside for HPO, to match the exported test models
random_test_split = kf_split[n_hpo_splits:]

# Regression metrics to compare random vs blocked
metric_names = ['r2', 'me', 'rmse', 'rrmse']

# Local metric coordinates for pair distances, adequate over extents of a single landscape
# (from 5_models.ipynb 'Annual disturbance effect')
# Calculated from the mean of the latitude feature of the model dataset
mean_latitude = np.radians(model_dataset['fea_latitude'].mean())
metres_per_degree_lat = 111132.9 - 559.8 * np.cos(2 * mean_latitude) + 1.2 * np.cos(4 * mean_latitude)
metres_per_degree_lon = 111412.8 * np.cos(mean_latitude) - 93.5 * np.cos(3 * mean_latitude)

# Exact footprint coordinates for distances, east then north
footprint_coords = np.stack([footprint_x * metres_per_degree_lon, footprint_y * metres_per_degree_lat], axis=1)

# Coordinates from latitude and longitude features used to assign spatial blocks
feature_coords = np.stack([model_dataset['fea_longitude'].to_numpy(dtype=np.float64) * metres_per_degree_lon,
                           model_dataset['fea_latitude'].to_numpy(dtype=np.float64) * metres_per_degree_lat], axis=1)


# Predict each held-out fold with its test model
def predict_test_folds(split, test_model_dir, x=None):
    if x is None:
        x = model_dataset_x
    fold_predictions = []
    for fold_index, (train_index, valid_index) in enumerate(split, start=1):
        model = xgb.Booster()
        model.load_model(join(test_model_dir, f"model_test_fold_{fold_index}.json"))
        prediction = model.predict(xgb.DMatrix(x.loc[valid_index], enable_categorical=True))
        fold_predictions.append(pd.DataFrame({
            'fold': fold_index,
            'observed': model_dataset_y.loc[valid_index].to_numpy(dtype=np.float64),
            'predicted': prediction.astype(np.float64)}, index=valid_index))
    return pd.concat(fold_predictions).sort_index()


# Mean of the per-fold scores, reported for cross-validation
def fold_scores(predictions):
    scores = [calculate_metrics(fold['observed'], fold['predicted']) for _, fold in predictions.groupby('fold')]
    return {m: np.mean([s[m] for s in scores]) for m in metric_names}

## Residual correlogram

In [ ]:
# Checks spatial autocorrelation of out-of-fold residuals from random cross-validation
# Moran's I calculated for pairs of footprints in each distance class, then compared
# with 95% interval from randomly permuting residual values across footprints

# Distance class edges (m)
correlogram_edges = [0, 100, 250, 500, 1000, 2000, 5000]

# Pairs are formed from random sample of focal footprints to all held-out footprints
# retaining every close neighbour while limiting memory use
correlogram_focal_sample = 10000
correlogram_permutations = 199

# Footprints per chunk when accumulating pairs
# Doesn't affect results, just memory use
correlogram_chunk = 1000

# Out-of-fold predictions from  exported random cross-validation test models
random_predictions = predict_test_folds(random_test_split, test_model_dir)
random_predictions['residual'] = random_predictions['observed'] - random_predictions['predicted']

# Confirm exported models reproduce recorded test scores
random_scores = fold_scores(random_predictions)
print(f"Random cross-validation test RMSE from exported models: {random_scores['rmse']:.2f}, "
      f"recorded: {model_description['score_validation (rmse) mean']:.2f}\n")


def residual_correlogram(residuals, coords, seed=1):
    rng = np.random.default_rng(seed)
    deviations = residuals - residuals.mean()
    variance = np.mean(deviations ** 2)
    coord_tree = cKDTree(coords)
    focal = rng.choice(residuals.size, size=min(correlogram_focal_sample, residuals.size), replace=False)

    # Accumulate pairs within outer distance class edge
    pair_i, pair_j, pair_class = [], [], []
    for chunk_start in range(0, focal.size, correlogram_chunk):
        chunk = focal[chunk_start:chunk_start + correlogram_chunk]
        neighbours = coord_tree.query_ball_point(coords[chunk], correlogram_edges[-1])
        neighbour_counts = np.fromiter((len(n) for n in neighbours), dtype=np.int64, count=chunk.size)
        j = np.concatenate([np.asarray(n, dtype=np.int64) for n in neighbours])
        i = np.repeat(chunk, neighbour_counts)
        not_self = i != j
        i, j = i[not_self], j[not_self]
        separation = np.linalg.norm(coords[i] - coords[j], axis=1)
        distance_class = np.clip(np.searchsorted(correlogram_edges, separation, side='left') - 1, 0, None)
        pair_i.append(i.astype(np.int32))
        pair_j.append(j.astype(np.int32))
        pair_class.append(distance_class.astype(np.int8))
    pair_i, pair_j, pair_class = np.concatenate(pair_i), np.concatenate(pair_j), np.concatenate(pair_class)
    n_classes = len(correlogram_edges) - 1
    pair_counts = np.bincount(pair_class, minlength=n_classes)

    # Moran's I with binary weights in each distance class
    def morans_i(values):
        return np.bincount(pair_class, weights=values[pair_i] * values[pair_j], minlength=n_classes) / pair_counts / variance

    observed = morans_i(deviations)
    permuted = np.array([morans_i(rng.permutation(deviations)) for _ in range(correlogram_permutations)])
    return pd.DataFrame({
        'distance_lower': correlogram_edges[:-1],
        'distance_upper': correlogram_edges[1:],
        'pairs': pair_counts,
        'morans_i': observed,
        'null_lower': np.percentile(permuted, 2.5, axis=0),
        'null_upper': np.percentile(permuted, 97.5, axis=0),
    })


correlogram_table = residual_correlogram(random_predictions['residual'].to_numpy(),
                                         footprint_coords[random_predictions.index])
# Number held-out footprints and run settings, for figure caption
correlogram_table['held_out_footprints'] = len(random_predictions)
correlogram_table['focal_footprints'] = min(correlogram_focal_sample, len(random_predictions))
correlogram_table['permutations'] = correlogram_permutations
correlogram_table.to_csv(join(spatial_dir, "residual_correlogram.csv"), index=False)

print(tabulate(correlogram_table.round(4), headers='keys', tablefmt='simple', showindex=False))

## Spatially blocked folds

In [ ]:
# Grid cell size (m) for spatial blocks, where footprints in the same cell are held out together
# 2 km is chosen as twice the distance past which residual autocorrelation becomes negligible
# (Moran's I < 0.01)
block_size = 2000

# Assign footprints to grid cells, use coordinates of the blocked cross-validation
block_columns = np.floor(feature_coords[:, 0] / block_size).astype(np.int64)
block_rows = np.floor(feature_coords[:, 1] / block_size).astype(np.int64)
_, block_id = np.unique(np.stack([block_columns, block_rows], axis=1), axis=0, return_inverse=True)
block_id = block_id.ravel()

# Stratify by same AGBD quantile bins and random state as the original random cross-validation,
# grouped by grid cell
blocked_split = list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=1).split(model_dataset_x, y_binned, groups=block_id))
blocked_test_split = blocked_split[n_hpo_splits:]

# Check target distribution and number of blocks in each test fold
fold_rows = []
for fold_index, (train_index, valid_index) in enumerate(blocked_test_split, start=1):
    y_fold = model_dataset_y.loc[valid_index]
    fold_rows.append({
        'fold': fold_index,
        'footprints': len(valid_index),
        'blocks': len(np.unique(block_id[valid_index])),
        'agbd_mean': y_fold.mean(),
        'agbd_p05': y_fold.quantile(0.05),
        'agbd_median': y_fold.median(),
        'agbd_p95': y_fold.quantile(0.95),
    })
blocked_folds_table = pd.DataFrame(fold_rows).round(1)
blocked_folds_table.to_csv(join(spatial_dir, f"blocked_folds_{block_size}m.csv"), index=False)
print(f"{len(np.unique(block_id)):,} grid cells of {block_size} m, median of {np.median(np.bincount(block_id)):.0f} footprints per cell")
print(tabulate(blocked_folds_table, headers='keys', tablefmt='simple', showindex=False))

## Blocked cross-validation

In [ ]:
# Blocked cross-validation with the same hyperparameters and number of test folds
# as the random cross-validation
blocked_metrics_path = join(spatial_dir, f"blocked_cv_metrics_{block_size}m.csv")

# Test model (from 5_models.ipynb 'Final model', 'Define and test')
predictor = XGBPredictor(**test_hyperparameters)

# Run model. Blocked test folds skipped if already run
if not exists(blocked_metrics_path):
    df_kfold_metrics = run_kfold(model_dataset, blocked_test_split, predictor, verbose=True)
    df_kfold_metrics.to_csv(blocked_metrics_path, index=False)
else:
    df_kfold_metrics = pd.read_csv(blocked_metrics_path)
    print(f"Blocked cross-validation was already run: {blocked_metrics_path}")

# Compare with random cross-validation in model_description.json
comparison_rows = []
for m in metric_names:
    comparison_rows.append({
        'metric': m,
        'random_test_mean': model_description[f"score_validation ({m}) mean"],
        'random_test_std': model_description[f"score_validation ({m}) std"],
        'blocked_test_mean': df_kfold_metrics[f"score_test ({m})"].mean(),
        'blocked_test_std': df_kfold_metrics[f"score_test ({m})"].std(),
        # Random test mean from exported test models
        'random_test_mean_exported_models': random_scores[m],
    })
cv_comparison_table = pd.DataFrame(comparison_rows)
cv_comparison_table.to_csv(join(spatial_dir, f"cv_comparison_{block_size}m.csv"), index=False)
print(tabulate(cv_comparison_table.round(3), headers='keys', tablefmt='simple', showindex=False))

## Nearest-neighbour distances

In [ ]:
# Distance from each held-out footprint to its nearest training footprint
nearest_distances = {}
for cv_name, split in [('Random', random_test_split), ('Blocked', blocked_test_split)]:
    fold_distances = []
    for train_index, valid_index in split:
        train_tree = cKDTree(footprint_coords[train_index])
        distances, _ = train_tree.query(footprint_coords[valid_index], k=1)
        fold_distances.append(distances)
    nearest_distances[cv_name] = np.concatenate(fold_distances)

nearest_distance_table = pd.DataFrame({'median_m': {cv_name: np.median(distances) for cv_name, distances in nearest_distances.items()}})

In [ ]:
# Distance of each forest pixel of the actual 2024 scenario from the nearest footprint
# The prediction task that the held-out distances of the cross-validation compare with
prediction_distance_scenario = "2024"
prediction_distance_polygon = "project_area.gpkg"

# Pixels per chunk when querying distances. Doesn't affect results, just memory use
prediction_distance_chunk = 1_000_000

# Scenario mask of the actual year, made in 6_scenarios.ipynb
scenario_mask_array = read_raster_as_array(join(scenario_masks_dir, f"{prediction_distance_scenario}.tif"))

# Forest pixels in the project area
prediction_rows, prediction_cols, _ = polygon_pixel_indices(join(polygons_dir, prediction_distance_polygon))
prediction_forest = scenario_mask_array[prediction_rows, prediction_cols] == 1
prediction_rows, prediction_cols = prediction_rows[prediction_forest], prediction_cols[prediction_forest]
scenario_mask_array = prediction_forest = None

# Pixel centre coordinates, converted to same metric coordinates as the footprints
template_ds = gdal.Open(template_tif_path)
geotransform = template_ds.GetGeoTransform()
template_ds = None
prediction_x = (geotransform[0] + (prediction_cols + 0.5) * geotransform[1]) * metres_per_degree_lon
prediction_y = (geotransform[3] + (prediction_rows + 0.5) * geotransform[5]) * metres_per_degree_lat
prediction_rows = prediction_cols = None

# Nearest footprint in the full dataset, the one used to train the original final model
footprint_tree = cKDTree(footprint_coords)
prediction_distances = np.empty(prediction_x.size)
for chunk_start in range(0, prediction_x.size, prediction_distance_chunk):
    chunk = slice(chunk_start, chunk_start + prediction_distance_chunk)
    prediction_distances[chunk], _ = footprint_tree.query(np.stack([prediction_x[chunk], prediction_y[chunk]], axis=1), k=1)
prediction_x = prediction_y = footprint_tree = None

# Add results to nearest-neighbour distance table
nearest_distance_table.loc['Prediction', 'median_m'] = np.median(prediction_distances)
nearest_distance_table.to_csv(join(spatial_dir, f"nearest_distances_{block_size}m.csv"))
print(f"Distances calculated for {prediction_distances.size:,} forest pixels of {prediction_distance_scenario} scenario.\n")
print(tabulate(nearest_distance_table.round(1), headers='keys', tablefmt='simple'))

## Figures

In [ ]:
# Plot settings (from 5_models.ipynb 'Annual disturbance effect')
dpi = 300
width = 950
height = 600
title_size = 21
axis_title_size = 17
axis_value_size = 15
legend_size = 17

# Marker colour for correlalilulelogram
correlogram_colour = '#d95f0e'


# Figure layout, following style of 5_models.ipynb 'Annual disturbance effect' plots
def style_figure(fig, title_text, x_title, y_title, x_type='-'):
    fig.update_layout(
        xaxis=dict(
            title=dict(text=x_title, font=dict(family="helvetica", size=axis_title_size, color="black")),
            tickfont=dict(family="helvetica", size=axis_value_size, color="black"),
            type=x_type, showgrid=True, gridcolor='lightgrey', gridwidth=0.5),
        yaxis=dict(
            title=dict(text=y_title, font=dict(family="helvetica", size=axis_title_size, color="black")),
            tickfont=dict(family="helvetica", size=axis_value_size, color="black"),
            showgrid=True, gridcolor='lightgrey', gridwidth=0.5, zeroline=True, zerolinecolor='grey'),
        legend=dict(x=0.99, y=0.99, xanchor='right', bgcolor='rgba(255,255,255,0.8)',
                    font=dict(family="helvetica", size=legend_size, color="black")),
        width=width, height=height, plot_bgcolor='white', paper_bgcolor='white', margin=dict(t=60, r=30),
        annotations=[dict(x=0, y=1.11, xref='paper', yref='paper', text=title_text, showarrow=False,
                          xanchor='left', align='left',
                          font=dict(family="helvetica", size=title_size, color="black", weight=600))])
    return fig


# Residual correlowlowlowgram with permutation interval
distance_labels = [f"{lower}-{upper}" for lower, upper in zip(correlogram_table['distance_lower'], correlogram_table['distance_upper'])]
fig_correlogram = go.Figure()
fig_correlogram.add_trace(go.Scatter(
    x=distance_labels + distance_labels[::-1],
    y=np.concatenate([correlogram_table['null_upper'], correlogram_table['null_lower'][::-1]]),
    fill='toself', fillcolor='rgba(128, 128, 128, 0.3)', line=dict(color='grey', width=1),
    name='95% permutation interval', hoverinfo='skip'))
fig_correlogram.add_trace(go.Scatter(
    x=distance_labels, y=correlogram_table['morans_i'], name="Moran's I", mode='lines+markers',
    line=dict(color='black', width=2), marker=dict(size=8, color=correlogram_colour, line=dict(color='black', width=1))))
style_figure(fig_correlogram, 'Residual spatial autocorrelation', 'Distance between footprints (m)', "Moran's I")

# Save png and svg
for filename, fig in [('residual_correlogram', fig_correlogram)]:
    fig.write_image(join(spatial_dir, f'{filename}.png'), scale=dpi / 96)
    fig.write_image(join(spatial_dir, f'{filename}.svg'), scale=dpi / 96)
    fig.show()
print(f"Spatial independence completed, gone to {spatial_dir}")

# Neutral covariates

In [ ]:
# Test bias from fixing GEDI covariates at values used for raster predictions
# Though the design and purpose of neutral covariates was actually to reduce bias,
# This can be verified here, again.
# Load neutral effect values for covariates from SHAP analysis (from 6_scenarios.ipynb, 'Feature stacks')
# These effects were actually used to minimise covariate bias when making spatial predictions.
# Otherwise, set a manual override

covariate_override = False

# Manual overrides (applied after SHAP values loaded)
covariate_overrides = {
    'fea_beam': 4,
    'fea_sensitivity': 0.98,
}

covariate_values = {}
feature_analysis_path = join(selected_model_shap_dir, 'shap_feature_analysis.csv')
if covariates_renamed:
    print(f"Covariates defined in model: {covariates_renamed}\n")
    if exists(feature_analysis_path):
        feature_analysis = pd.read_csv(feature_analysis_path)
        print("Neutral effect values from SHAP analysis:")
        for cov in covariates_renamed:
            row = feature_analysis[feature_analysis['Dataset name'] == cov]
            if not row.empty:
                neutral_val = row['Neutral_Effect_Value'].values[0]
                shap_at_neutral = row['SHAP_at_Neutral'].values[0]
                covariate_values[cov] = neutral_val
                print(f"  {cov}: {neutral_val:.4f} (SHAP at neutral: {shap_at_neutral:.4f})")
            else: print(f"  {cov}: not found in SHAP analysis")
    else: print("SHAP feature analysis not found. Set the values in covariate_overrides.")

    # Apply overrides and verify completeness
    if covariate_override:
      print(f"\nManual overrides applied: {covariate_overrides}")
      covariate_values.update(covariate_overrides)
    missing = [c for c in covariates_renamed if c not in covariate_values]
    assert not missing, f"Missing covariate values: {missing}. Set in covariate_overrides."
print()

# Replace recorded covariates with neutral values. Categorical values are category codes (from 0
# position in category list), as passed to the model in feature stacks
model_dataset_x_neutral = model_dataset_x.copy()
# Neutral values applied, saved with the results
neutral_values = {}
for cov, value in covariate_values.items():
    if cov in categorical_columns:
        categories = model_dataset_x[cov].cat.categories
        model_dataset_x_neutral[cov] = pd.Categorical.from_codes(np.full(len(model_dataset_x_neutral), int(value)), categories=categories)
        # Report original value of the category applied
        category_value = categories[int(value)]
        original_value = {int(v): k for k, v in categorical_features_mappings[cov].items()}.get(int(category_value))
        print(f"{cov}: code {int(value)}, category {category_value}, original value {original_value}")
        neutral_values[f"neutral_{cov.removeprefix('fea_')}"] = original_value
    else:
        model_dataset_x_neutral[cov] = np.float32(value)
        print(f"{cov}: {value}")
        neutral_values[f"neutral_{cov.removeprefix('fea_')}"] = value
print()

# Bias as the change in mean predicted AGBD, using random cross-validation predictions from
# 'Spatial independence', 'Residual correlogram'
neutral_predictions = predict_test_folds(random_test_split, test_model_dir, x=model_dataset_x_neutral)
neutral_table = pd.DataFrame([{
    'mean_prediction_shift': neutral_predictions['predicted'].mean() - random_predictions['predicted'].mean(),
    'mean_prediction_shift_percent': 100 * (neutral_predictions['predicted'].mean() / random_predictions['predicted'].mean() - 1),
    **neutral_values,
}])
neutral_table.to_csv(join(neutral_dir, "neutral_covariates.csv"), index=False)
print(tabulate(neutral_table.round(3), headers='keys', tablefmt='simple', showindex=False))

# Old-growth hold-out

## Hold-out polygons

In [ ]:
# Tests the robustness of the old-growth proxy - whether parts of the proxy itself are predicted
# as old-growth when they're held out. Nod to reviewers, this test is a very good idea.
# Purported old-growth in each hold-out area is unlabelled, i.e. treated as non-reference forest
# with an uncertain intactness in training and prediction, and the area is given its own
# management unit feature (as forest reserves were in the Malaysia case study). A new model for each
# hold-out area also tests how predictions depend on that particular part of the old-growth feature.

# Old-growth polygon divided into parts of equal area, West to East
# Use for larger protected areas
part_polygon = "taman_negara.gpkg"
part_prefix = "taman_negara_part"
n_parts = 5

# Old-growth polygons held out entirely, for smaller, discrete protected areas
whole_holdouts = {
    "tengku_hassanal": "tengku_hassanal_wr.gpkg",
}

# Sample polygon compared with the hold-outs, from names printed in 'Sample polygons'
comparison_polygon_name = "ASARTR phase 1"

# Paths
holdout_polygons_dir = join(holdout_dir, "polygons")
makedirs(holdout_polygons_dir, exist_ok=True)
holdouts_path = join(holdout_dir, "old_growth_holdouts.gpkg")

# Pixels and cell areas of part polygon
part_rows, part_cols, part_cell_area_ha = polygon_pixel_indices(join(polygons_dir, part_polygon))

# Area within polygon of each template column, accumulated West to East
cumulative_column_area = np.cumsum(np.bincount(part_cols - part_cols.min(), weights=part_cell_area_ha))
# Part edges are first column after each equal share of the total area is reached
part_edges = [0] + [int(np.searchsorted(cumulative_column_area, cumulative_column_area[-1] * part / n_parts)) + 1
                    for part in range(1, n_parts)] + [cumulative_column_area.size]
part_edges = [part_cols.min() + edge for edge in part_edges]
part_rows = part_cols = part_cell_area_ha = None

# Part polygons are clipped between the longitudes of the column edges
# Each pixel is assigned to only one part when rasterised
template_ds = gdal.Open(template_tif_path)
geotransform = template_ds.GetGeoTransform()
template_ds = None
part_polygon_read = gpd.read_file(join(polygons_dir, part_polygon)).to_crs("EPSG:4326").dissolve()
part_polygon_geometry = part_polygon_read.geometry.iloc[0]
part_bounds = part_polygon_read.total_bounds
holdout_rows = []
for part in range(n_parts):
    west = geotransform[0] + part_edges[part] * geotransform[1]
    east = geotransform[0] + part_edges[part + 1] * geotransform[1]
    part_geometry = part_polygon_geometry.intersection(box(west, part_bounds[1] - 1, east, part_bounds[3] + 1))
    if part_geometry.geom_type == 'GeometryCollection':
        part_geometry = gpd.GeoSeries([g for g in part_geometry.geoms if g.geom_type in ['Polygon', 'MultiPolygon']]).unary_union
    holdout_rows.append({'name': f"{part_prefix}_{part + 1}", 'geometry': part_geometry})
for holdout_name, holdout_polygon in whole_holdouts.items():
    holdout_geometry = gpd.read_file(join(polygons_dir, holdout_polygon)).to_crs("EPSG:4326").dissolve().geometry.iloc[0]
    holdout_rows.append({'name': holdout_name, 'geometry': holdout_geometry})
holdouts_gdf = gpd.GeoDataFrame(holdout_rows, crs="EPSG:4326")

# Each hold-out saved as an individual polygon for burning, and then together for mapping
holdout_paths = {}
for i, holdout_name in enumerate(holdouts_gdf['name']):
    holdout_path = join(holdout_polygons_dir, f"{holdout_name}.gpkg")
    if not exists(holdout_path): holdouts_gdf.iloc[[i]].to_file(holdout_path, driver="GPKG")
    holdout_paths[holdout_name] = holdout_path
if not exists(holdouts_path): holdouts_gdf.to_file(holdouts_path, driver="GPKG")

# Hold-out areas from rasterised polygons, used for polygon statistics
comparison_area_km2 = polygon_data[comparison_polygon_name]['cell_area_ha'].sum() / 100
holdout_area_rows = []
for holdout_name, holdout_path in holdout_paths.items():
    _, _, holdout_cell_area_ha = polygon_pixel_indices(holdout_path)
    holdout_area_rows.append({'hold-out': holdout_name, 'area_km2': holdout_cell_area_ha.sum() / 100,
                              f'percent_of_{comparison_polygon_name}': 100 * holdout_cell_area_ha.sum() / 100 / comparison_area_km2})
print(f"{comparison_polygon_name}: {comparison_area_km2:,.0f} km2")
print(tabulate(pd.DataFrame(holdout_area_rows).round(1), headers='keys', tablefmt='simple', showindex=False))
print(f"\nHold-out polygons are here: {holdouts_path}")

## Relabelled old-growth features

In [ ]:
# Each hold-out burned into the old-growth binary as 0, and into a new land-use binary as 1
# This is used as an additional management unit feature, with edge effects

# Old-growth land-use base name, used for 'oldgrowth_feature' in 6_scenarios.ipynb
old_growth_base = "lu_old-growth_protected_areas"

# Literature supports edge effects up to 120m (from 3a_features_scenario.ipynb, 'Binary feature edge effects')
edge_effect_distance = 120

# Old-growth feature paths
old_growth_feature_name = f"{old_growth_base}_edge_distance"
old_growth_binary_path = join(feature_binary_dir, f"{old_growth_base}_binary.tif")
old_growth_edge_distance_path = join(feature_edge_effects_dir, f"{old_growth_feature_name}.tif")

# Directory and feature paths of hold-outs
holdout_features = {}
for holdout_name in holdout_paths:
    holdout_model_dir = join(holdout_dir, holdout_name)
    makedirs(holdout_model_dir, exist_ok=True)
    holdout_unit_base = f"lu_{holdout_name}"
    holdout_features[holdout_name] = {
        'dir': holdout_model_dir,
        'old_growth_binary': join(holdout_model_dir, f"{old_growth_base}_binary.tif"),
        'old_growth_edge_distance': join(holdout_model_dir, f"{old_growth_feature_name}.tif"),
        'unit_feature_name': f"{holdout_unit_base}_edge_distance",
        'unit_binary': join(holdout_model_dir, f"{holdout_unit_base}_binary.tif"),
        'unit_edge_distance': join(holdout_model_dir, f"{holdout_unit_base}_edge_distance.tif"),
    }

# Arrays shared by all hold-outs, used to create features and check for change
old_growth_binary_array = read_raster_as_array(old_growth_binary_path)
old_growth_edge_distance_array = read_raster_as_array(old_growth_edge_distance_path)
cell_area_array = read_raster_as_array(cell_area_path)

# Set all template values to 0 in preparation (from 3a_features_scenario.ipynb, 'LU polygon binary features')
template_mask_array = gdal.Open(template_tif_path).ReadAsArray()
template_mask_array = np.zeros_like(template_mask_array, dtype=np.int16)

# Hold-out features
for holdout_name, holdout_path in holdout_paths.items():
    paths = holdout_features[holdout_name]

    # Relabelled old-growth feature
    if not exists(paths['old_growth_edge_distance']):
        # Copy old-growth binary and burn hold-out as '0'
        export_array_as_tif(old_growth_binary_array.astype(np.int16), paths['old_growth_binary'], dtype=gdal.GDT_Int16)
        burn_polygon_to_raster(paths['old_growth_binary'], holdout_path, fixed=True, fixed_value=0, all_touched=False)
        binary_array = read_raster_as_array(paths['old_growth_binary'])
        # Generate edge distance and local density features (from 3a_features_scenario.ipynb 'Binary feature edge effects')
        signed_distance, local_density = edge_effects(binary_array, 'binary', cell_size_x_path, cell_size_y_path, threshold_metres=edge_effect_distance)
        # Only edge distance exported, local density not a model feature
        export_array_as_tif(signed_distance, paths['old_growth_edge_distance'], dtype=gdal.GDT_Int16)
        print(f"{holdout_name}: unlabelled old-growth edge distance feature created.")
    else: print(f"{holdout_name}: unlabelled old-growth edge distance feature already exists.")

    # Hold-out management unit feature
    if not exists(paths['unit_edge_distance']):
        export_array_as_tif(template_mask_array, paths['unit_binary'], dtype=gdal.GDT_Int16)
        # Burn value '1' where overlapping with land use polygon (from 3a_features_scenario.ipynb, 'LU polygon binary features')
        burn_polygon_to_raster(paths['unit_binary'], holdout_path, fixed=True, fixed_value=1, all_touched=False)
        binary_array = read_raster_as_array(paths['unit_binary'])
        # Edge distance and local density features (from 3a_features_scenario.ipynb, 'Binary feature edge effects')
        signed_distance, local_density = edge_effects(binary_array, 'binary', cell_size_x_path, cell_size_y_path, threshold_metres=edge_effect_distance)
        # Only edge distance exported, local density was not used a model feature for management units
        export_array_as_tif(signed_distance, paths['unit_edge_distance'], dtype=gdal.GDT_Int16)
        print(f"{holdout_name}: management unit edge distance feature created.")
    else: print(f"{holdout_name}: management unit edge distance feature already exists.")

    # Check change against original feature
    unlabelled_edge_distance_array = read_raster_as_array(paths['old_growth_edge_distance'])
    changed_pixels = old_growth_edge_distance_array != unlabelled_edge_distance_array
    unit_pixels = read_raster_as_array(paths['unit_edge_distance']) > 0
    print(f"  Old-growth feature: {changed_pixels.sum():,} pixels changed ({cell_area_array[changed_pixels].sum() / 1e6:.1f} km2)")
    print(f"  Management unit: {unit_pixels.sum():,} pixels inside ({cell_area_array[unit_pixels].sum() / 1e6:.1f} km2)")
    binary_array = signed_distance = local_density = None
    unlabelled_edge_distance_array = changed_pixels = unit_pixels = None

old_growth_binary_array = old_growth_edge_distance_array = cell_area_array = template_mask_array = None

## Resample training features

In [ ]:
# Sample the original, relabelled and hold-out management unit features at footprint locations
# Exact footprint coordinates used, each footprint samples the same pixel as in 4_datasets.ipynb
# Changed columns of each hold-out saved, so models can be trained in a new runtime

# Set number of threads for raster pixel sampling (from 4_datasets.ipynb 'Sample feature pixels')
# Lower this if runtime is unstable.
n_threads = os.cpu_count()-1

geom_x, geom_y = footprint_x, footprint_y
old_growth_column = f"fea_{old_growth_feature_name}"
holdout_training_paths = {holdout_name: join(holdout_features[holdout_name]['dir'], "training_features.pkl") for holdout_name in holdout_paths}

# Checked only if any hold-out not sampled
if not all(exists(path) for path in holdout_training_paths.values()):
    # Resampling original feature must reproduce model dataset
    original_sample = pd.DataFrame(index=model_dataset.index)
    sample_raster_values(original_sample, old_growth_edge_distance_path, geom_x, geom_y, feature=True, n_threads=n_threads)
    n_mismatch = int((original_sample[old_growth_column].to_numpy() != model_dataset[old_growth_column].to_numpy()).sum())
    assert n_mismatch == 0, f"{n_mismatch:,} footprints do not reproduce original feature, check coordinates."
    print("Resampled original feature matches model dataset.\n")
    original_sample = None

# Each hold-out, skipped if sampled already
for holdout_name, holdout_training_path in holdout_training_paths.items():
    if exists(holdout_training_path):
        print(f"{holdout_name}: training features already exist.")
        continue
    paths = holdout_features[holdout_name]
    unit_column = f"fea_{paths['unit_feature_name']}"
    unlabelled_sample = pd.DataFrame(index=model_dataset.index)
    sample_raster_values(unlabelled_sample, paths['old_growth_edge_distance'], geom_x, geom_y, feature=True, n_threads=n_threads)
    unit_sample = pd.DataFrame(index=model_dataset.index)
    sample_raster_values(unit_sample, paths['unit_edge_distance'], geom_x, geom_y, feature=True, n_threads=n_threads)

    # Relabelled old-growth values and hold-out management unit feature
    holdout_training = pd.DataFrame(index=model_dataset.index)
    holdout_training[old_growth_column] = unlabelled_sample[old_growth_column].to_numpy().astype(model_dataset[old_growth_column].dtype)
    holdout_training[unit_column] = unit_sample[unit_column].to_numpy().astype(model_dataset[old_growth_column].dtype)
    holdout_training.to_pickle(holdout_training_path)

    changed_footprints = holdout_training[old_growth_column] != model_dataset[old_growth_column]
    print(f"{holdout_name}: {changed_footprints.sum():,} of {len(model_dataset):,} footprints changed ({100 * changed_footprints.mean():.2f}%), "
          f"{(holdout_training[unit_column] > 0).sum():,} inside the management unit")
    unlabelled_sample = unit_sample = holdout_training = changed_footprints = None

## Train hold-out models

In [ ]:
# Train on full dataset with the hyperparameters and seed used in original final model
# (from 5_models.ipynb, 'Final model', 'Train and export with full dataset')

# Each hold-out model, skipped if it already exist
for holdout_name, holdout_training_path in holdout_training_paths.items():
    paths = holdout_features[holdout_name]
    holdout_model_path = join(paths['dir'], "model.json")
    if exists(holdout_model_path):
        print(f"{holdout_name}: a hold-out model already exists.")
        continue

    # Replace old-growth feature wit relabelled values and add hold-out management unit feature
    unit_column = f"fea_{paths['unit_feature_name']}"
    holdout_training = pd.read_pickle(holdout_training_path)
    model_dataset_holdout = model_dataset.copy()
    model_dataset_holdout[old_growth_column] = holdout_training[old_growth_column].to_numpy()
    model_dataset_holdout[unit_column] = holdout_training[unit_column].to_numpy()

    # Feature order of hold-out model, sorted as in 5_models.ipynb with covariates
    holdout_model_features = sorted([f for f in selected_features if f not in covariates_renamed] + [unit_column]) + covariates_renamed

    # Create DMatrix objects
    # Hold-out dataset and features replace model_dataset_x
    dtrain_final = xgb.DMatrix(model_dataset_holdout[holdout_model_features], model_dataset_y, enable_categorical=True)

    # Train model on full dataset
    predictor = XGBPredictor(**final_hyperparameters)
    params_final = predictor.get_params()
    params_final['eval_metric'] = eval_metric
    # sklearn wrapper params handled separately via num_boost_round and DMatrix
    for key in ['n_estimators', 'enable_categorical', 'missing']: params_final.pop(key, None)
    # verbose_eval set to False, because several models are trained in one cell
    model_final = xgb.train(params_final,
                            dtrain_final,
                            num_boost_round=final_hyperparameters['n_estimators'],
                            verbose_eval=False)

    # Export model
    model_final.save_model(holdout_model_path)
    print(f"{holdout_name}: model training and 'model.json' export complete.")

    # The hold-out model description, with its features for prediction
    holdout_description = {
        "selected_model": selected_model,
        "holdout_name": holdout_name,
        "holdout_path": holdout_paths[holdout_name],
        "old_growth_feature": old_growth_column,
        "holdout_unit_feature": unit_column,
        "footprints_changed": int((holdout_training[old_growth_column] != model_dataset[old_growth_column]).sum()),
        "selected_features": holdout_model_features,
        "hyperparameters": str(final_hyperparameters),
    }
    with open(join(paths['dir'], "model_description.json"), "w") as f:
        f.write(json.dumps(holdout_description, indent=2))
    holdout_training = model_dataset_holdout = dtrain_final = model_final = None

## Scenario predictions

In [ ]:
# Check existing tile parameters, (from 6_scenarios.ipynb 'Scenario predictions')
template_tile_list = [f for f in os.listdir(tile_templates_dir) if f.endswith('.tif') and f.startswith('template_tile')]
n_tiles = len(template_tile_list)
assert n_tiles > 0, "There are no template tiles. Run the template tiles section, even if only one is created."
if n_tiles == 1: print(f"# There is 1 template tile.\n")
if n_tiles > 1: print(f"# There are {n_tiles} template tiles.\n")

# Collect available scenarios from the feature stack tiles directory
scenario_stacks_list = []
for scenario in os.listdir(feature_stacks_dir):
    stack_files = [f for f in os.listdir(join(feature_stacks_dir, scenario)) if f.startswith('feature_stack_')]
    if len(stack_files) == n_tiles:
        scenario_stacks_list.append(scenario)

print("scenarios_to_predict = [")
for scenario in sorted(scenario_stacks_list):
  print(f'  "{scenario}",')
print("]")

In [ ]:
# Scenario predictions of original and hold-out models (from 6_scenarios.ipynb 'Scenario predictions')
scenarios_to_predict = [
  "2024",
  "2024_undisturbed_since_oldgrowth_1",
  "2024_undisturbed_since_oldgrowth_2",
]

# Prediction raster precision. GEDI AGBD can be set to 0, any higher is
# spurious due to the wide prediction intervals of the source data.
raster_precision = 0

# Actual scenario and merged old-growth baseline
actual_scenario = "2024"
baseline_scenario = "2024_undisturbed_since_oldgrowth"

# Forest reserve feature with the scenario treatment that each hold-out management unit uses
# e.g. minimum edge distance in old-growth v1 (see 'oldgrowth_redundant_features', in 6_scenarios.ipynb)
reference_unit_feature = "lu_tekam_edge_distance"

# The original model + one model for each hold-out
models_to_predict = {'original': selected_model_json}
for holdout_name in holdout_paths: models_to_predict[holdout_name] = join(holdout_features[holdout_name]['dir'], "model.json")

# Prediction and tile cache directories of hold-out test
holdout_predictions_dir = join(holdout_dir, "scenario_predictions")
tile_prediction_cache_dir = join(holdout_dir, "tile_prediction_cache")
makedirs(holdout_predictions_dir, exist_ok=True)
makedirs(tile_prediction_cache_dir, exist_ok=True)

# Old-growth feature in scenario feature lists, and position of reference unit in feature stacks
old_growth_scenario_feature = f"{basename(feature_edge_effects_dir)}/{old_growth_feature_name}"
model_stack_features = [f[4:] for f in selected_features if f not in covariates_renamed]
assert reference_unit_feature in model_stack_features, f"{reference_unit_feature} is not a model feature."
reference_unit_position = model_stack_features.index(reference_unit_feature)

# Detect GPU availability and set predictor type. Note that XGBoost inference is
# not much faster on GPU than CPU, and transferring large feature stacks from
# CPU to GPU memory can actually make it much slower. TPU is not used, but the
# TPU Colab runtime can provide more memory and more CPU workers.
try:
    test_array = cupy.array([1, 2, 3])
    del test_array
    predictor_type = 'gpu_predictor'
    gpu_id, use_gpu = 0, True
    print("GPU detected and accessible - using GPU to load the feature stack and to predict.")
except:
    predictor_type = 'cpu_predictor'
    gpu_id, use_gpu = -1, False
    print("GPU not accessible - using CPU prediction")
xgb.set_config(verbosity=0, use_rmm=use_gpu)

# Check existing tile parameters
template_tile_list = [file for file in os.listdir(tile_templates_dir)
                     if file.endswith('.tif') and file[:13] == 'template_tile']
n_tiles = len(template_tile_list)
assert n_tiles > 0, "There are no template tiles. Run the template tiles section, even if only one is created."
template_tile = gdal.Open(join(tile_templates_dir,'template_tile_1.tif'))
template_tile_x = template_tile.GetRasterBand(1).XSize
template_tile = None
print(f"There are {n_tiles} template tiles.")

template_base_path = template_tif_path

# Template geotransform, for row offset of each tile
template_geotransform = gdal.Open(template_tif_path).GetGeoTransform()

# Load each model once, with features, and for hold-out models the position of management unit
# and relabelled old-growth feature
model_setups = {}
for model_label, model_path in models_to_predict.items():
  model_features = selected_features
  model_setup = {}
  if model_label != 'original':
    paths = holdout_features[model_label]
    with open(join(paths['dir'], "model_description.json")) as holdout_description_json:
      model_features = json.load(holdout_description_json)["selected_features"]
    model_setup['unit_position'] = model_features.index(f"fea_{paths['unit_feature_name']}")
    model_setup['old_growth_ds'] = gdal.Open(paths['old_growth_edge_distance'])

  # Build feature_types list matching selected_features order
  feature_types = []
  for feat in model_features:
      if feat in categorical_columns:
          feature_types.append('c')
      else:
          feature_types.append('q')

  # Select appropriate predictor type and set feature_types
  XGBPredictor = xgb.XGBRegressor()
  XGBPredictor.load_model(model_path)
  XGBPredictor.set_params(predictor=predictor_type, feature_types=feature_types)
  if use_gpu: XGBPredictor.set_params(device='cuda:0')

  model_setup['features'] = model_features
  model_setup['predictor'] = XGBPredictor
  model_setups[model_label] = model_setup
print(f"{len(model_setups)} models loaded.")

# Progress tracking
scenario_progress_index = 0
scenario_progress_label = widgets.Label(f"Scenario progress: {scenario_progress_index}/{len(scenarios_to_predict)}")
display(scenario_progress_label)
tile_progress_index, tile_progress_label = 0, widgets.Label(value=f"Tile progress: 0 / {n_tiles}")
display(tile_progress_label)

# Model progress in each tile
model_progress_label = widgets.Label(f"Model progress: 0/{len(model_setups)}")
display(model_progress_label)

# Loop through each scenario
for scenario in scenarios_to_predict:
  # Models still to predict for this scenario. Each tile feature stack is loaded once and shared by
  # all models, with the hold-out columns changed on a copy for each hold-out model
  models_remaining = []
  for model_label in models_to_predict:
    scenario_prediction_path = join(holdout_predictions_dir, f"{scenario}__{model_label}.tif")
    # Skip oldgrowth _1 and _2 scenarios if merged file already exists
    if '_oldgrowth_1' in scenario or '_oldgrowth_2' in scenario or '_oldgrowth_recovery_1' in scenario or '_oldgrowth_recovery_2' in scenario:
      merged_scenario = scenario.replace('_1', '').replace('_2', '')
      merged_prediction_path = join(holdout_predictions_dir, f"{merged_scenario}__{model_label}.tif")
      if exists(merged_prediction_path):
        print(f"Merged file exists, skipping: {scenario}__{model_label}")
        continue
    if not exists(scenario_prediction_path): models_remaining.append(model_label)

  if models_remaining:
    # Hold-out models: old-growth column to relabel if scenario uses the actual feature, and
    # management unit source, the hold-out feature where the reference unit uses its actual feature
    # otherwise same alternative feature (e.g. alternative/minimum_edge_distance)
    scenario_features = pd.Series.tolist(pd.read_csv(join(scenarios_model_dir, f"{scenario}.csv")).iloc[:,0])
    stack_features = [f for f in scenario_features if basename(f) not in covariates]
    swap_column = stack_features.index(old_growth_scenario_feature) if old_growth_scenario_feature in stack_features else None
    reference_unit_entry = stack_features[reference_unit_position]
    holdout_unit_datasets = {}
    for model_label in models_remaining:
      if model_label == 'original': continue
      if basename(reference_unit_entry) == reference_unit_feature: holdout_unit_source = holdout_features[model_label]['unit_edge_distance']
      else: holdout_unit_source = join(features_dir, f"{reference_unit_entry}.tif")
      holdout_unit_datasets[model_label] = gdal.Open(holdout_unit_source)

    scenario_feature_stack_dir = join(feature_stacks_dir, scenario)
    n_stacks = len([f for f in os.listdir(scenario_feature_stack_dir) if f.startswith('feature_stack_')])
    # Tile cache directory each model
    tile_cache_dirs = {model_label: join(tile_prediction_cache_dir, f"{scenario}__{model_label}") for model_label in models_remaining}
    for tile_cache_scenario_dir in tile_cache_dirs.values(): makedirs(tile_cache_scenario_dir, exist_ok=True)
    tile_progress_index = 0
    for tile_count in range(1, n_stacks + 1):
      scenario_tile_filename = f"scenario_tile_{tile_count}.tif"
      # Models without this tile in cache
      tile_models = [m for m in models_remaining if scenario_tile_filename not in os.listdir(tile_cache_dirs[m])]
      if tile_models:
        # Load template tile parameters
        template_tile_path = join(tile_templates_dir, f"template_tile_{tile_count}.tif")
        template_tile = gdal.Open(template_tile_path)
        template_tile_y = template_tile.GetRasterBand(1).YSize
        template_tile_x = template_tile.GetRasterBand(1).XSize
        # Row offset of tile in the template
        tile_row_offset = int(round((template_tile.GetGeoTransform()[3] - template_geotransform[3]) / template_geotransform[5]))
        template_tile = None
        n_pixels = template_tile_y * template_tile_x
        # Load feature stack and valid indices
        stack_path = join(scenario_feature_stack_dir, f"feature_stack_{scenario}_{tile_count}.npy")
        indices_path = join(scenario_feature_stack_dir, f"valid_indices_{scenario}_{tile_count}.npy")
        # Loaded fully into memory, as stack is shared by all models
        feature_stack = np.load(stack_path)
        valid_indices = np.load(indices_path)
        n_valid = len(valid_indices)

        for model_index, model_label in enumerate(tile_models, start=1):
          # Update model progress before prediction
          model_progress_label.value = f"Model progress: {model_index} / {len(tile_models)} ({model_label})"
          model_setup = model_setups[model_label]
          XGBPredictor = model_setup['predictor']
          # Handle empty tiles (no valid pixels)
          if n_valid == 0:
            if raster_precision == 0:
              prediction_tile = np.full((template_tile_y, template_tile_x), nodatavalue, dtype=np.int16)
            else:
              prediction_tile = np.full((template_tile_y, template_tile_x), nodatavalue, dtype=np.float32)
            export_array_as_tif(prediction_tile, join(tile_cache_dirs[model_label], scenario_tile_filename),
                                template=template_tile_path)
            prediction_tile = None
            continue
          # Hold-out models: replace old-growth column with the relabelled feature, and insert
          # management unit column in hold-out model's feature order, on copy of the shared stack
          model_stack = feature_stack
          if model_label != 'original':
            model_stack = feature_stack.copy()
            if swap_column is not None:
              feature_strip = model_setup['old_growth_ds'].GetRasterBand(1).ReadAsArray(0, tile_row_offset, template_tile_x, template_tile_y)
              model_stack[:, swap_column] = feature_strip.ravel(order='C')[valid_indices]
            holdout_unit_strip = holdout_unit_datasets[model_label].GetRasterBand(1).ReadAsArray(0, tile_row_offset, template_tile_x, template_tile_y)
            model_stack = np.insert(model_stack, model_setup['unit_position'], holdout_unit_strip.ravel(order='C')[valid_indices].astype(np.float32), axis=1)
            feature_strip = holdout_unit_strip = None
            assert model_stack.shape[1] == len(model_setup['features']), "Feature stack columns do not match the hold-out model features."
          # Load stack to GPU if available
          if use_gpu:
            try: model_stack = cupy.asarray(model_stack)
            except Exception as e:
                if "Memory allocation error" in str(e) or "Out of memory" in str(e):
                    print("GPU memory insufficient, switching to CPU for this tile.")
                    cupy.get_default_memory_pool().free_all_blocks()
                    gc.collect()
                    XGBPredictor.set_params(device='cpu', predictor='cpu_predictor')
                else: raise
          # Predict, though terminate runtime if GPU prediction fails
          try:
              prediction = XGBPredictor.predict(model_stack)
          except Exception as e:
              if "Memory allocation error" in str(e) or "Out of memory" in str(e):
                  print("GPU memory insufficient for prediction. Terminating runtime to save compute units, restart with TPU.")
                  runtime.unassign()
              else: raise
          model_stack = None
          # Reconstruct full tile from valid indices (C-order to match stacking)
          if raster_precision == 0:
            prediction_flat = np.full(n_pixels, nodatavalue, dtype=np.int16)
            prediction_flat[valid_indices] = np.round(prediction).astype(np.int16)
          else:
            prediction_flat = np.full(n_pixels, nodatavalue, dtype=np.float32)
            prediction_flat[valid_indices] = np.round(prediction, raster_precision)
          prediction = None
          prediction_tile = prediction_flat.reshape((template_tile_y, template_tile_x), order='C')
          prediction_flat = None
          # Export prediction array as .tif
          export_array_as_tif(prediction_tile, join(tile_cache_dirs[model_label], scenario_tile_filename),
                            template=template_tile_path)
          prediction_tile = None
        # Release shared tile before loading next
        feature_stack = valid_indices = None
        if use_gpu: cupy.get_default_memory_pool().free_all_blocks()
      tile_progress_index += 1
      tile_progress_label.value = f"Tile progress: {tile_progress_index} / {n_stacks}"

    # Merge tiles of each model
    for model_label in models_remaining:
      tile_cache_scenario_dir = tile_cache_dirs[model_label]
      scenario_prediction_path = join(holdout_predictions_dir, f"{scenario}__{model_label}.tif")
      # Prepare empty array for merging tiles
      prediction_array = np.empty((0, template_tile_x))
      # Ensure tiles are in the correct order
      tile_files = sorted([f for f in os.listdir(tile_cache_scenario_dir) if f.endswith('.tif')],
                          key=lambda x: int(x.split('_')[-1].split('.')[0]))
      # Read each tile .tif as an array, stack, then export as a .tif
      for fname in tile_files:
          tile = gdal.Open(join(tile_cache_scenario_dir, fname))
          tile_array = tile.ReadAsArray()
          prediction_array = np.vstack((prediction_array, tile_array))
          tile = None
      if raster_precision == 0: prediction_array = np.round(prediction_array, raster_precision).astype(np.int16)
      else: prediction_array = np.round(prediction_array, raster_precision)
      export_array_as_tif(prediction_array, scenario_prediction_path, template=template_base_path)
      print(f"Prediction exported: {basename(scenario_prediction_path)}")
      # Delete scenario tile cache directory
      shutil.rmtree(tile_cache_scenario_dir)
    holdout_unit_datasets = None
  # Update scenario progress
  scenario_progress_index += 1
  scenario_progress_label.value = f"Scenario progress: {scenario_progress_index}/{len(scenarios_to_predict)}"
model_setups = None
print("\nScenario predictions complete.")

# Merge old-growth versions by taking maximum values.
# Version 1 uses old-growth proxy for pre-Landsat undisturbed forest.
# Version 2 removes all disturbance without the proxy.
# Taking maximum avoids underestimation where proxy may not capture all oldgrowth characteristics.

oldgrowth_v1_files = [f for f in os.listdir(holdout_predictions_dir)
                      if ('_oldgrowth_recovery_1__' in f or '_undisturbed_since_oldgrowth_1__' in f)
                      and f.endswith('.tif')]

if not oldgrowth_v1_files: print("No oldgrowth version 1 predictions found.")
else:
  for v1_file in oldgrowth_v1_files:
    # Construct version 2 and merged filenames
    base_name = v1_file.split('__')[0][:-1] + '2'  # Replace '1' with '2'
    rest_of_name = '__' + v1_file.split('__')[1]
    v2_file = f"{base_name}{rest_of_name}"
    merged_file = v1_file.replace('_1__', '__')
    merged_path = join(holdout_predictions_dir, merged_file)
    # Skip if merged file already exists
    if exists(merged_path):
      print(f"Merged file already exists: {merged_file}")
      continue
    v1_path = join(holdout_predictions_dir, v1_file)
    v2_path = join(holdout_predictions_dir, v2_file)
    # Merge or copy depending on version 2 existence
    if exists(v2_path):
      print(f"Merging oldgrowth versions for {v1_file.split('__')[0]}...")
      oldgrowth_1 = gdal.Open(v1_path)
      oldgrowth_1_array = oldgrowth_1.ReadAsArray()
      oldgrowth_1 = None
      oldgrowth_2 = gdal.Open(v2_path)
      oldgrowth_2_array = oldgrowth_2.ReadAsArray()
      oldgrowth_2 = None
      merged_array = np.maximum(oldgrowth_1_array, oldgrowth_2_array)
      export_array_as_tif(merged_array, merged_path, template=template_base_path)
      merged_array = oldgrowth_1_array = oldgrowth_2_array = None
      print(f"Merged version exported: {merged_file}")
      # Delete originals
      os.remove(v1_path)
      os.remove(v2_path)
    else:
      shutil.copy2(v1_path, merged_path)
      print(f"Version 2 not found, copied version 1: {merged_file}")
      os.remove(v1_path)

  print("\nOldgrowth merging complete.")

## Percentage loss

In [ ]:
# From 8_comparisons.ipynb 'Intactness','Percentage loss'

percentage_loss_precision = 0

# Percentage loss quantifies AGBD change relative to baseline: (actual / baseline) * 100 - 100.
# Values >0 indicate AGBD gain, which (within scenario constraints) are artefacts of
# model instability. These are clipped to 0.

# Percentage loss of each model
holdout_percentage_loss_dir = join(holdout_dir, "percentage_loss")
makedirs(holdout_percentage_loss_dir, exist_ok=True)

# Global function: round array (from 8_comparisons.ipynb)
def round_array(arr, precision):
    rounded = np.round(arr, precision)
    return rounded.astype(np.int16) if precision == 0 else rounded

# Calculate percentage loss from actual and baseline arrays
def calculate_percentage(actual, baseline, nodata):
    actual = np.where(actual == nodata, np.nan, actual).astype(np.float64)
    baseline = np.where((baseline == nodata) | (baseline == 0), np.nan, baseline).astype(np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return (actual / baseline) * 100 - 100

# Each model, skipped if its percentage loss exists
for model_label in models_to_predict:
    percentage_path = join(holdout_percentage_loss_dir, f"percentage_loss__{baseline_scenario}__{model_label}.tif")
    if exists(percentage_path):
        print(f"Percentage loss already exists: {basename(percentage_path)}")
        continue

    # Load single prediction rasters from scenarios dir
    # Predictions of each model in the hold-out directory
    baseline_path_in = join(holdout_predictions_dir, f"{baseline_scenario}__{model_label}.tif")
    actual_path_in = join(holdout_predictions_dir, f"{actual_scenario}__{model_label}.tif")
    baseline_arr = read_raster_as_array(baseline_path_in)
    actual_arr = read_raster_as_array(actual_path_in)
    template_path = baseline_path_in

    # Define masks
    actual_nodata_mask = actual_arr == nodatavalue
    baseline_nodata_mask = baseline_arr == nodatavalue
    both_valid = ~baseline_nodata_mask & ~actual_nodata_mask
    deforestation_mask = ~baseline_nodata_mask & actual_nodata_mask
    zero_baseline_mask = both_valid & (baseline_arr == 0)

    # Calculate percentage loss
    percentage_arr = calculate_percentage(actual_arr, baseline_arr, nodatavalue)
    percentage_arr = np.where(baseline_nodata_mask, nodatavalue, percentage_arr)

    # Deforestation: -100% loss
    percentage_arr[deforestation_mask] = -100

    # Zero baseline: 0% loss
    percentage_arr[zero_baseline_mask] = 0

    # Clip percentage to [-100, 0]
    percentage_arr[both_valid] = np.clip(percentage_arr[both_valid], -100, 0)

    # Round outputs
    percentage_arr = round_array(percentage_arr, percentage_loss_precision)

    # Export outputs
    export_array_as_tif(percentage_arr, percentage_path, template=template_path)
    print(f"Percentage loss exported: {basename(percentage_path)}")

print("Percentage loss calculations complete.")

In [ ]:
# Composite percentage loss for map. Each hold-out shows the model in which it was held
# out, and the prioritisation area shows e per-pixel mean of all the hold-out models

# Prioritisation area mask
mask_polygon = 'forest_reserves.gpkg'

# Percentage loss of each hold-out model, read once for hold-outs and the prioritisation area
composite_path = join(holdout_percentage_loss_dir, f"percentage_loss__{baseline_scenario}__composite.tif")
prioritisation_rows, prioritisation_cols, _ = polygon_pixel_indices(join(polygons_dir, mask_polygon))
percentage_sum = np.zeros(len(prioritisation_rows), dtype='float64')
valid_count = np.zeros(len(prioritisation_rows), dtype='int16')
holdout_values = {}
composite_array = None
for holdout_name, holdout_path in holdout_paths.items():
    holdout_rows, holdout_cols, _ = polygon_pixel_indices(holdout_path)
    holdout_percentage_array = read_raster_as_array(join(holdout_percentage_loss_dir, f"percentage_loss__{baseline_scenario}__{holdout_name}.tif"))
    if composite_array is None:
        composite_array = np.full(holdout_percentage_array.shape, nodatavalue, dtype='float32')
    prioritisation_values = holdout_percentage_array[prioritisation_rows, prioritisation_cols].astype('float64')
    prioritisation_valid = prioritisation_values != nodatavalue
    percentage_sum[prioritisation_valid] += prioritisation_values[prioritisation_valid]
    valid_count += prioritisation_valid
    holdout_values[holdout_name] = (holdout_rows, holdout_cols, holdout_percentage_array[holdout_rows, holdout_cols])
    holdout_percentage_array = None

# Prioritisation area as mean where every hold-out model has value. Deforested pixels
# identical in every model, so stay at -100%
all_valid = valid_count == len(holdout_paths)
composite_array[prioritisation_rows[all_valid], prioritisation_cols[all_valid]] = percentage_sum[all_valid] / len(holdout_paths)
percentage_sum = valid_count = all_valid = None

# Each hold-out from own model
for holdout_name, (holdout_rows, holdout_cols, values) in holdout_values.items():
    composite_array[holdout_rows, holdout_cols] = values
holdout_values = None
export_array_as_tif(composite_array, composite_path)
composite_array = None
print(f"Composite percentage loss sent to: {composite_path}. Almost done now.")

## Statistics

In [ ]:
# Percentage loss statistics (from 9_statistics.ipynb, 'Intactness statistics')
# Calculates area-weighted percentage loss statistics per polygon, for extant forest pixels only

# Baseline scenario of percentage loss rasters
baseline_name = baseline_scenario

# Sample polygon names printed in the 'Sample polygons' section (need to check back)
# Prioritisation area polygon for overall change in percentage loss
prioritisation_area_name = "Prioritisation area"

# Management units in the prioritisation area, used for rank correlation
prioritisation_units = [
    "ASARTR phase 1",
    "ASARTR phase 2",
    "Berkelah Jerantut forest reserve",
    "Berkelah Kuantan forest reserves",
    "Berkelah Temerloh forest reserve",
    "Remen Chereh forest reserves",
    "Tekai Tembeling forest reserves",
    "Tekam forest reserve",
    "Yong forest reserves",
    "Yong Lipis forest reserves",
]

# Hold-outs added to sample polygons, statistics saved in hold-out directory
for holdout_name, holdout_path in holdout_paths.items():
    holdout_pixel_rows, holdout_pixel_cols, holdout_cell_area_ha = polygon_pixel_indices(holdout_path)
    polygon_data[holdout_name] = {'rows': holdout_pixel_rows, 'cols': holdout_pixel_cols, 'cell_area_ha': holdout_cell_area_ha}

polygon_names = list(polygon_data.keys())
n_polygons = len(polygon_names)

# Progress tracking
progress_max = len(models_to_predict)
progress_index = 0
progress_label = widgets.Label(f"Percentage loss raster progress: {progress_index}/{progress_max}")
display(progress_label)

# Statistics for each model, from percentage loss raster
model_statistics = {}
for model_label in models_to_predict:
    percentage_raster_path = join(holdout_percentage_loss_dir, f"percentage_loss__{baseline_name}__{model_label}.tif")

    # Load rasters
    percent_ds = gdal.Open(percentage_raster_path)
    percent_array = percent_ds.ReadAsArray().astype('float64')
    percent_nodata = percent_ds.GetRasterBand(1).GetNoDataValue()
    percent_ds = None

    # Initialise output dataframe
    columns = [
        "Name",
        "Percentage loss (extant) mean",
        "Percentage loss (extant) stdev",
    ]
    df_percentage_stats = pd.DataFrame(columns=columns)

    # Inner loop: one row per polygon
    for polygon_name in polygon_names:
        rows = polygon_data[polygon_name]['rows']
        cols = polygon_data[polygon_name]['cols']
        cell_area_ha = polygon_data[polygon_name]['cell_area_ha']

        # Take values from percentage loss raster
        percent_values = percent_array[rows, cols]
        pct_valid = (percent_values != percent_nodata)

        # Skip polygon when no valid data
        if not np.any(pct_valid):
            new_row = pd.DataFrame([{
                'Name': polygon_name,
                'Percentage loss (extant) mean': None,
                'Percentage loss (extant) stdev': None,
            }], dtype=object)
            df_percentage_stats = pd.concat([df_percentage_stats, new_row], ignore_index=True)
            continue

        # Percentage loss statistics
        # Extant mask: valid pixels with loss > -100% (extant forest)
        pct_extant_mask = pct_valid & (percent_values > -100)

        # Calculate areas in hectares
        pct_extant_area = cell_area_ha[pct_extant_mask].sum(dtype='float64')

        # Calculate percentage loss statistics
        if pct_extant_area > 0:
            # Extant: only pixels with loss > -100%
            extant_pct_vals = percent_values[pct_extant_mask]
            extant_pct_weights = cell_area_ha[pct_extant_mask]

            # Area-weighted mean
            percent_extant_mean = np.sum(extant_pct_vals * extant_pct_weights, dtype='float64') / pct_extant_area

            # Area-weighted stdev
            variance = np.sum(extant_pct_weights * (extant_pct_vals - percent_extant_mean)**2, dtype='float64') / pct_extant_area
            percent_extant_std = np.sqrt(variance)
        else:
            # No extant forest
            percent_extant_mean = percent_extant_std = None

        # Build output row
        new_row_dict = {
            'Name': polygon_name,
            'Percentage loss (extant) mean': percent_extant_mean,
            'Percentage loss (extant) stdev': percent_extant_std,
        }
        new_row = pd.DataFrame([new_row_dict], dtype=object)
        df_percentage_stats = pd.concat([df_percentage_stats, new_row], ignore_index=True)

    df_percentage_stats = df_percentage_stats.set_index('Name')
    # Statistics for each model, kept for hold-out summary
    model_statistics[model_label] = df_percentage_stats.astype(float)

    progress_index += 1
    progress_label.value = f"Percentage loss raster progress: {progress_index}/{progress_max}"

# Tables of each statistic, polygons as rows, models as columns
def model_table(statistic):
    return pd.DataFrame({model_label: stats[statistic] for model_label, stats in model_statistics.items()})
percent_extant = model_table('Percentage loss (extant) mean')
percent_extant_std = model_table('Percentage loss (extant) stdev')

# Mean extant percentage loss of each management unit across hold-out models (standard deviation
# is between models, NOT pixels), will use on map labels
holdout_names = list(holdout_paths)
holdout_units_table = percent_extant.loc[prioritisation_units, holdout_names]
unit_percentage = pd.DataFrame({
    'Percentage loss (extant) mean': holdout_units_table.mean(axis=1),
    'Percentage loss (extant) stdev': holdout_units_table.std(axis=1),
    'Percentage loss (extant) min': holdout_units_table.min(axis=1),
    'Percentage loss (extant) max': holdout_units_table.max(axis=1),
    'Percentage loss (extant) original model': percent_extant.loc[prioritisation_units, 'original'],
})
# Each hold-out model's value, for ranges and ranks across refits
unit_percentage = unit_percentage.join(holdout_units_table.add_prefix('Percentage loss (extant) '))
unit_percentage.to_csv(join(holdout_dir, "old_growth_holdout_unit_percentage_loss.csv"))
print(tabulate(unit_percentage.round(2), headers='keys', tablefmt='simple'))
print()

# Each hold-out area from own model, comparison polygon from the same model, and
# sensitivity of prioritisation area results to the hold-out.
# Units ranked by percentage loss, extant only, deforestation is identical in every model anyways
original_units = percent_extant.loc[prioritisation_units, 'original']
holdout_table_rows = []
for holdout_name in holdout_paths:
    holdout_units = percent_extant.loc[prioritisation_units, holdout_name]
    rho = spearmanr(original_units, holdout_units)[0]
    # Pairs of units that swap order relative to original model, with their difference in original model
    swap_differences = [abs(original_units[a] - original_units[b])
                        for i, a in enumerate(prioritisation_units) for b in prioritisation_units[i + 1:]
                        if np.sign(original_units[a] - original_units[b]) != np.sign(holdout_units[a] - holdout_units[b])]
    holdout_table_rows.append({
        'Hold-out': holdout_name,
        'Area (km2)': polygon_data[holdout_name]['cell_area_ha'].sum() / 100,
        'Percentage loss (extant) mean': percent_extant.loc[holdout_name, holdout_name],
        'Percentage loss (extant) stdev': percent_extant_std.loc[holdout_name, holdout_name],
        f'{comparison_polygon_name}: percentage loss (extant) mean': percent_extant.loc[comparison_polygon_name, holdout_name],
        f'{prioritisation_area_name}: percentage loss (extant) change': percent_extant.loc[prioritisation_area_name, holdout_name] - percent_extant.loc[prioritisation_area_name, 'original'],
        f'Spearman rho ({len(prioritisation_units)} units)': rho,
        'Least degraded unit': holdout_units.idxmax(),
        'Swapped unit pairs': len(swap_differences),
        'Swapped unit pairs: largest difference in original model': max(swap_differences, default=0),
    })
holdout_table = pd.DataFrame(holdout_table_rows)
holdout_table.to_csv(join(holdout_dir, "old_growth_holdout_by_holdout.csv"), index=False)

# Mean, standard deviation (between hold-outs, NOT pixels) and range for hold-outs
summary_columns = [c for c in holdout_table.columns if c not in ['Hold-out', 'Area (km2)', 'Percentage loss (extant) stdev', 'Least degraded unit']]
holdout_summary = pd.DataFrame({
    'mean': holdout_table[summary_columns].mean(),
    'stdev': holdout_table[summary_columns].std(),
    'min': holdout_table[summary_columns].min(),
    'max': holdout_table[summary_columns].max(),
})
holdout_summary.to_csv(join(holdout_dir, "old_growth_holdout_summary.csv"))

print(tabulate(holdout_table.set_index('Hold-out').T.round(3), headers='keys', tablefmt='simple'))
print()
print(tabulate(holdout_summary.round(3), headers='keys', tablefmt='simple'))

# Disconnect runtime

In [ ]:
# Useful for stopping background execution
runtime.unassign()